# Lecture 1: The Storage and Serialization Bridge

## 1. Escaping the Small Files Problem
When dealing with millions of small files (like individual images), the I/O latency equation for independent files is dominated by seek time:
$$T_{total} = N \times \left(T_{seek} + \frac{S_{file}}{B_{network}}\right)$$
With $N=60000$ and $T_{seek}$ dominating network latency, GPUs stay idle waiting for data roundtrips.

The amortized equation for contiguous sharding (like WebDataset Tar archives) is:
$$T_{total} \approx T_{seek\_archive} + \frac{N \times S_{file}}{B_{network}}$$
Here, we pay the seek penalty only once per archive, massively increasing throughput.


## 2. Interactive S3 Exploration
We can verify that our generated shards are correctly stored in our simulated S3 (MinIO) bucket using `boto3` pagination.

In [ ]:
import boto3

# Initialize S3 Client matching our MinIO configuration
s3 = boto3.client(
    's3',
    endpoint_url='http://127.0.0.1:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

# Use pagination to list objects in the cifar-streaming bucket
paginator = s3.get_paginator('list_objects_v2')
try:
    pages = paginator.paginate(Bucket='cifar-streaming')
    print("Files in 'cifar-streaming':")
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']][:5]: # Print first 5
                print(f"- {obj['Key']} ({obj['Size'] / 1024 / 1024:.2f} MB)")
            print("... (truncated)")
except Exception as e:
    print(f"Error accessing bucket (Did you create it?): {e}")

## 3. Streaming Data Pipeline
Now we build our `WebDataset` pipeline to read directly from the streaming endpoint. We decode the images directly into memory without writing to standard disk.

In [ ]:
import webdataset as wds
import matplotlib.pyplot as plt

# The URL prefix to our local MinIO shards. Replace the range if you generated different shards.
url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Construct WebDataset pipeline
dataset = (
    wds.WebDataset(url)
    .decode("rgb")
    .to_tuple("png", "cls")
)

# Fetch a single batch of 16 images
images = []
labels = []

try:
    for idx, (img, label) in enumerate(dataset):
        images.append(img)
        labels.append(label)
        if idx == 15:
            break

    # Visualize the 4x4 grid
    if images:
        fig, axes = plt.subplots(4, 4, figsize=(8, 8))
        axes = axes.flatten()

        for i in range(len(images)):
            axes[i].imshow(images[i])
            axes[i].set_title(f"Class: {labels[i]}")
            axes[i].axis("off")

        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"Could not read from MinIO stream. Ensure MinIO is running and bucket contains data! {e}")